# RSI(3) > 30 Bounce Backtester
# max stocks: 50
# Regime Filter: Close > 89-EMA
# Setup: RSI(3) crossing above 30
# Exits: +5% TP, -3% SL, 8 Days Max Hold

In [1]:
# 1. Imports and Setup
import pandas as pd
import numpy as np
import os
import glob
import random
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML
import warnings

warnings.filterwarnings('ignore')
display(HTML("<style>.container { width:100% !important; }</style>"))
sns.set_theme(style="darkgrid")

DATA_DIR = r'D:\0dot1_Aug_2016_master\data\mstock_mtf_daily_data'


In [2]:
# 2. Data Loading & Indicator Calculations
def calc_rsi(series, period):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

all_data = []
files = glob.glob(os.path.join(DATA_DIR, '*.csv'))
random.seed(42) 
random.shuffle(files)

loaded = 0
for file_path in files:
    if loaded >= 50: 
        break
    symbol = os.path.basename(file_path).replace('.csv', '')
    try:
        df = pd.read_csv(file_path).dropna(subset=['Close'])
        df['Date'] = pd.to_datetime(df['Date'])
        df = df.sort_values('Date').reset_index(drop=True)
        
        if len(df) < 300: 
            continue
            
        latest_price = df['Close'].iloc[-1]
        if not (90 <= latest_price <= 600):
            continue
            
        # Indicators
        df['EMA_89'] = df['Close'].ewm(span=89, adjust=False).mean()
        df['RSI_3'] = calc_rsi(df['Close'], 3)
        
        # Crossover Logic: Yesterday RSI < 30 AND Today RSI >= 30
        df['Setup_Signal'] = (df['Close'] > df['EMA_89']) & (df['RSI_3'] >= 30) & (df['RSI_3'].shift(1) < 30)
        
        # Keep recent 2 years for backtest
        df = df.tail(500).reset_index(drop=True)
        
        df['Symbol'] = symbol
        all_data.append(df)
        loaded += 1
    except Exception as e:
        pass

data = pd.concat(all_data, ignore_index=True)
print(f"Loaded and calculated {len(data['Symbol'].unique())} stocks.")


Loaded and calculated 50 stocks.


In [3]:
# 3. Backtesting Engine
trades = []

for symbol in data['Symbol'].unique():
    df = data[data['Symbol'] == symbol].reset_index(drop=True)
    
    in_trade = False
    entry_price = 0
    entry_date = None
    days_held = 0
    
    for i, row in df.iterrows():
        if pd.isna(row['EMA_89']):
            continue
            
        # Manage Open Trades
        if in_trade:
            days_held += 1
            exit_triggered = False
            exit_reason = ""
            
            # Calculate Profit
            profit_pct = (row['Close'] - entry_price) / entry_price
            
            # Hard Rules
            if profit_pct >= 0.05:
                exit_triggered = True
                exit_reason = "Take Profit (+5%)"
            elif profit_pct <= -0.03:
                exit_triggered = True
                exit_reason = "Stop Loss (-3%)"
            elif days_held >= 8:
                exit_triggered = True
                exit_reason = "Time Stop (8 Days)"
            
            if exit_triggered:
                trades.append({
                    'Symbol': symbol,
                    'Entry_Date': entry_date,
                    'Exit_Date': row['Date'],
                    'Entry_Price': entry_price,
                    'Exit_Price': row['Close'],
                    'Profit_Pct': profit_pct * 100,
                    'Days_Held': days_held,
                    'Reason': exit_reason
                })
                in_trade = False
                days_held = 0
                
        # Look for Entries
        if not in_trade and row['Setup_Signal']:
            in_trade = True
            entry_price = row['Close']
            entry_date = row['Date']

trades_df = pd.DataFrame(trades)
print(f"Backtest Complete. Found {len(trades_df)} total trades.")


Backtest Complete. Found 911 total trades.


In [4]:
# 4. Scoreboard and Analysis
if len(trades_df) > 0:
    total_trades = len(trades_df)
    winners = trades_df[trades_df['Profit_Pct'] > 0]
    losers = trades_df[trades_df['Profit_Pct'] <= 0]
    
    win_rate = len(winners) / total_trades * 100
    avg_profit = trades_df['Profit_Pct'].mean()
    avg_win = winners['Profit_Pct'].mean() if len(winners) > 0 else 0
    avg_loss = losers['Profit_Pct'].mean() if len(losers) > 0 else 0
    avg_hold = trades_df['Days_Held'].mean()
    
    reason_counts = trades_df['Reason'].value_counts()
    
    html = f'''
    <h1 style="color: #2e6c80;">RSI(3) > 30 Scoreboard</h1>
    <p>Tested on 50 stocks over the last ~2 years.</p>
    <ul>
        <li><b>Total Trades:</b> {total_trades}</li>
        <li><b>Win Rate:</b> {win_rate:.2f}%</li>
        <li><b>Net Avg Profit:</b> {avg_profit:.2f}%</li>
        <li><b>Average Win:</b> +{avg_win:.2f}%</li>
        <li><b>Average Loss:</b> {avg_loss:.2f}%</li>
        <li><b>Avg Hold Time:</b> {avg_hold:.1f} Days</li>
    </ul>
    <h3>Exit Breakdown:</h3>
    '''
    display(HTML(html))
    display(reason_counts)
    
    display(HTML("<h3>Latest Trades Sample:</h3>"))
    display(trades_df.tail(15).sort_values('Exit_Date', ascending=False))
else:
    print("No trades found matching criteria.")


Reason
Stop Loss (-3%)       413
Time Stop (8 Days)    258
Take Profit (+5%)     240
Name: count, dtype: int64

,Symbol,Entry_Date,Exit_Date,Entry_Price,Exit_Price,Profit_Pct,Days_Held,Reason
910,GOLD1,2026-06-15,2026-06-19,125.120003,120.589996,-3.620529,4,Stop Loss (-3%)
909,GOLD1,2026-06-02,2026-06-08,129.660004,125.070000,-3.540031,4,Stop Loss (-3%)
908,GOLD1,2026-05-19,2026-06-01,131.919998,128.720001,-2.425710,8,Time Stop (8 Days)
907,GOLD1,2026-04-30,2026-05-13,124.699997,132.940002,6.607863,8,Take Profit (+5%)
906,GOLD1,2026-04-15,2026-04-27,126.500000,125.279999,-0.964428,8,Time Stop (8 Days)
905,GOLD1,2026-03-10,2026-03-16,133.460007,128.169998,-3.963741,4,Stop Loss (-3%)
904,GOLD1,2026-02-19,2026-03-02,128.279999,139.509995,8.754284,7,Take Profit (+5%)
903,GOLD1,2026-02-04,2026-02-05,130.660004,126.059998,-3.520592,1,Stop Loss (-3%)
902,GOLD1,2026-01-02,2026-01-14,112.330002,118.510002,5.501647,8,Take Profit (+5%)
901,GOLD1,2025-12-05,2025-12-17,107.639999,110.800003,2.935715,8,Time Stop (8 Days)
